In [ ]:
import pandas as pd
import os
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_community.llms import Ollama
from dotenv import load_dotenv
from langchain_community.callbacks import get_openai_callback

from neo4j import GraphDatabase
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.schema import get_schema

d:\GitHub\muict_thai_legal_RAG_GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Knowledge Graph Construction

In [5]:
llm = ChatOpenAI(api_key=os.getenv("openrouter_API_key") ,temperature=0, model="openai/gpt-4o-mini", base_url="https://openrouter.ai/api/v1")

# llm = Ollama(model="llama3", ) # Tested with llama3


# allowed_nodes = [
#     "Law",              # ชื่อกฎหมาย (เช่น ป.พ.พ., พ.ร.บ. คุ้มครองข้อมูลส่วนบุคคล)
#     "Section",          # มาตรา
#     "LegalElement",     # องค์ประกอบ (เช่น "เจตนา", "ประมาท", "โดยทุจริต")
#     "Exception",        # ข้อยกเว้น (สำคัญมากสำหรับกฎหมายไทย)
#     "Penalty",          # บทลงโทษ
#     "Subject",          # ตัวละครในกฎหมาย (เช่น เจ้าพนักงาน, ผู้รับมอบอำนาจ, ผู้บริโภค)
#     "LegalConsequence"  # ผลทางกฎหมาย (เช่น โมฆะ, ระงับ, สิ้นสุด)
# ]

# 3. กำหนด Relationships ให้ครอบคลุมตรรกะทางกฎหมาย
# allowed_relationships = [
#     ("Law", "CONTAINS", "Section"),
#     ("Section", "HAS_ELEMENT", "LegalElement"),  # มาตรานี้มีองค์ประกอบอะไรบ้าง
#     ("Section", "HAS_EXCEPTION", "Exception"),   # มาตรานี้มีข้อยกเว้นคืออะไร
#     ("Section", "PUNISHED_BY", "Penalty"),       # บทลงโทษของมาตรานี้
#     ("Section", "CROSS_REFERENCE", "Section"),   # การอ้างถึงมาตราอื่น (ใช้แทน CITES/REFERENCES)
#     ("LegalElement", "LEADS_TO", "LegalConsequence"), # องค์ประกอบครบถ้วนนำไปสู่ผลทางกฎหมาย
#     ("Subject", "HAS_DUTY", "LegalElement"),     # บุคคลมีหน้าที่ต้องทำอะไร
#     ("Exception", "OVERRIDES", "Section")        # ข้อยกเว้นที่ไปหักล้างหลักการในมาตรา
# ]

# 4. กำหนด Properties (เก็บเฉพาะข้อมูลที่เป็น Metadata จริงๆ)
# ข้อมูลที่เป็นเนื้อหาสำคัญควรแยกเป็น Node เพื่อความ Rich ของ GraphRAG
# node_properties = ["law_name", "section_num", "text"]

llm_transformer = LLMGraphTransformer(llm=llm,) # Model ต้องรองรับ tool เพื่อรองรับ Node properties
# llm_transformer = LLMGraphTransformer(llm=llm,allowed_nodes=allowed_nodes, allowed_relationships=allowed_relationships, node_properties=node_properties,) # Model ต้องรองรับ tool เพื่อรองรับ Node properties

In [4]:
from neo4j import GraphDatabase
load_dotenv(dotenv_path="../.env")

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE_NAME = os.getenv("NEO4J_DATABASE")


with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    # 2. Verify connectivity to ensure a connection can be established
    driver.verify_connectivity()
    print("Connection established successfully.")
    
graph = Neo4jGraph(url=URI, username=USERNAME, password=PASSWORD,database=DATABASE_NAME, enhanced_schema=True)

Connection established successfully.


In [ ]:
df = pd.read_parquet('../data/processed/nitibench_cleaned_2026-03-17.parquet')
df.head()

,law_name,section_content,section_num
0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,132
1,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,1598/5
2,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,876
3,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,1030
4,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,849


In [18]:
df[df['section_content'].str.contains('หมวด')].head().to_csv('test.csv')

In [19]:
len(df)

3833

In [20]:
df.iloc[:10,:]

,law_name,section_content,section_num
0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,132
1,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,1598/5
2,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,876
3,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,1030
4,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,849
5,พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558,พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558 มาต...,9
6,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 565\nการเช่าถ...,565
7,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/26\nผู้เ...,1598/26
8,พระราชบัญญัติการบัญชี พ.ศ. 2543,พระราชบัญญัติการบัญชี พ.ศ. 2543 มาตรา 21 ในการ...,21
9,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 193\nในกรณีดั...,193


In [ ]:
# text = ""
# col = df.columns.to_list()
# for i in range(10):
#     sub_text = ""
#     for j in range(len(col)-1):
#         sub_text += df.iloc[i, j]
#     text += sub_text
#     text += "\n"
# print(text)

พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน
ประมวลกฎหมายแพ่งและพาณิชย์ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5
ถ้าผู้อยู่ในปกครองรู้จักผิดชอบและมีอายุไม่ต่ำกว่าสิบห้าปีบริบูรณ์เมื่อผู้ปกครองจะทำกิจการใดที่สำคัญ ให้ปรึกษาหารือผู้อยู่ในปกครองก่อนเท่าที่จะทำได้
การที่ผู้อยู่ในปกครองได้ยินยอมด้วยนั้นหาคุ้มผู้ปกครองให้พ้นจากความรับผิดไม่
ประมวลกฎหมายแพ่งและพาณิชย์ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876
ถ้าผู้รับประกันภัยต้องคำพิพากษาให้เป็นคนล้มละลาย ผู้เอาประกันภัยจะเรียกให้หาประกันอันสมควรให้แก่ตนก็ได้ หรือจะบอกเลิกสัญญาเสียก็ได้
ถ้าผู้เอาประกันภัยต้องคำพิพากษาให้เป็นคนล้มละลาย ท่านให้ใช้วิธีเดียวกันนี้บังคับตามควรแก่เรื่อง แต่กระนั้นก็ดี ถ้าเบี้ยประกันภัยได้ส่งแล้วเต็มจำนวนเพื่ออายุประกันภัยเป็นระยะเวล

In [58]:
def batch_document_append(df,start=0, end=len(df)):
    documents = []
    for index, row in df.iloc[start:end,:].iterrows():
        # สร้างเนื้อหาที่มีบริบทชัดเจนในตัวเอง
        content = f"กฎหมาย: {row['law_name']}\nมาตรา: {row['section_num']}\nเนื้อหา: {row['section_content']}"
        
        # เก็บ Metadata เพื่อใช้ในภายหลัง
        metadata = {
            "law_name": row['law_name'],
            "section_num": str(row['section_num']),
        }
        documents.append(Document(page_content=content, metadata=metadata))
    return documents

documents = batch_document_append(df, start=0, end=50)
documents

[Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132'}, page_content='กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nมาตรา: 132\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
 Document(metadata={'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 'section_num': '1598/5'}, page_content='กฎหมาย: ประมวลกฎหมายแพ่งและพาณิชย์\nมาตรา: 1598/5\nเนื้อหา: ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู้อยู่ในปกครองรู้จักผิดชอบและมีอายุไม่ต่ำกว่าสิบห้าปีบริบูรณ์เมื่อผู้ปกครองจะทำกิจการใดที่สำคัญ ให้ปรึกษาหารือผู้อยู่ในปกครองก่อนเท่าที่จะทำได้\nการที่ผู้อยู่ในปกครองได้ยินยอมด้วยนั้นหาคุ้มผู้ปกครองให้พ้นจากความรับผิดไม่'),
 Document(metadata={'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 'section_num': '87

In [24]:
import time 

with get_openai_callback() as cb:
    start_time = time.perf_counter()

    # documents = [Document(page_content=text)]
    graph_documents = llm_transformer.convert_to_graph_documents(documents)

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    print(f"Elapsed time: {elapsed_time:.4f} seconds")
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")


Elapsed time: 270.3324 seconds
Total Tokens: 0
Prompt Tokens: 0
Completion Tokens: 0
Total Cost (USD): $0.0


In [25]:
for node in graph_documents[0].nodes:
    print(node)

id='ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54' type='Entity' properties={}
id='มาตรา 132' type='Clause' properties={}
id='ใบอนุญาต' type='Document' properties={}
id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546' type='Law' properties={}
id='มาตรา 54' type='Clause' properties={}


In [26]:
for relationship in graph_documents[0].relationships:
    print(relationship)

source=Node(id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', type='Law', properties={}) target=Node(id='มาตรา 132', type='Clause', properties={}) type='DEFINED_BY' properties={}
source=Node(id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', type='Law', properties={}) target=Node(id='ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54', type='Entity', properties={}) type='REGULATES' properties={}
source=Node(id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', type='Law', properties={}) target=Node(id='ใบอนุญาต', type='Document', properties={}) type='REGULATES' properties={}
source=Node(id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', type='Law', properties={}) target=Node(id='มาตรา 54', type='Clause', properties={}) type='REGULATES' properties={}


In [ ]:
graph.add_graph_documents(graph_documents, include_source=True)

## Querying the graph

### Text to Cypher Approach

In [62]:
schema = graph.get_schema
schema

'Node properties:\n- **Law**\n  - `id`: STRING Available options: [\'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\', \'ประมวลกฎหมายแพ่งและพาณิชย์\', \'พระราชบัญญัติการบัญชี พ.ศ. 2543\', \'กฎหมายว่าด้วยล้มละลาย\', \'ประมวลรัษฎากร\', \'กฎหมาย\']\n- **Clause**\n  - `id`: STRING Example: "มาตรา 132"\n- **Laws**\n  - `id`: STRING Available options: [\'พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558\', \'พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558 มาตรา 9\', \'ประมวลกฎหมายแพ่งและพาณิชย์\']\n- **Section**\n  - `id`: STRING Example: "มาตรา 1598/5"\n- **Person**\n  - `id`: STRING Example: "ผู้เป็นหุ้นส่วนคนนั้น"\n- **Action**\n  - `id`: STRING Available options: [\'เอาทรัพย์สินออกแบ่ง\', \'การก่อสร้างอย่างอื่น\', \'การเพิกถอนการสมรส\']\n- **Content**\n  - `id`: STRING Available options: [\'เนื้อหา\']\n- **Right**\n  - `id`: STRING Available options: [\'เรียกให้หาประกันอันสมควร\', \'สิทธิเหนือทรัพย์สินอันเป็นวัตถุแห่งสิทธิที่เป็นหลั\', \'สิทธิเลิกสัญญา\', \'ไม่ยอมชำระหนี้\', \'ชำระหนี้\', \'จำนองที่ดิน

In [6]:
from langchain_core.prompts.prompt import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """
Task: Generate a Neo4j Cypher statement based on the provided schema to answer Thai legal questions.

Schema Context: {schema}
- Main Node: `Document` (Contains properties: `text`, `law_name`, `section_num`, `id`)
- Other Nodes: `Law`, `Clause`, `Section`, `Entity`, `Action`, `Party`, `Right`, etc.
- Primary Relationship: `(:Document)-[:MENTIONS]->(n)` (Use this to find related concepts or entities mentioned in a specific legal section)

Instructions:
1. Always start by filtering the `Document` node using `law_name` and `section_num` if provided in the question.
2. If the user asks for "บทลงโทษ" or specific details, use `d.text CONTAINS "..."` to filter the content.
3. [Crucial] To find related context or neighbors, traverse from the `Document` node using the `[:MENTIONS]` relationship to other labels like `Entity`, `Action`, or `Right`.
4. Use `OPTIONAL MATCH` for traversals to ensure the main document is returned even if no specific neighbors are found.
5. Return the main legal text along with a list of mentioned entities/concepts.

Example 1: (Filtering and Traversal)
Question: "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร และเกี่ยวข้องกับใครหรืออะไรบ้าง"
Cypher:
MATCH (d:Document)
WHERE d.law_name CONTAINS "สัญญาซื้อขายล่วงหน้า" 
  AND d.text CONTAINS "ประกอบกิจการ" 
  AND d.text CONTAINS "ไม่ได้รับใบอนุญาต"
OPTIONAL MATCH (d)-[:MENTIONS]->(neighbor)
RETURN d.law_name, d.section_num, d.text, 
       collect(DISTINCT {{label: labels(neighbor)[0], id: neighbor.id}}) AS Mentions

Example 2: (Specific Section Search)
Question: "มาตรา 193 ของกฎหมายแพ่งและพาณิชย์ กล่าวถึงเรื่องอะไร"
Cypher:
MATCH (d:Document {{section_num: "193"}})
WHERE d.law_name CONTAINS "แพ่งและพาณิชย์"
OPTIONAL MATCH (d)-[:MENTIONS]->(m)
RETURN d.text, collect(labels(m)) AS RelatedCategories

Note: Do not include any text except the generated Cypher statement.
The question is: {question}
"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
)

In [7]:
# 1. นิยาม Prompt สำหรับการสรุปคำตอบ (QA)
QA_TEMPLATE = """Task: คุณคือทนายความผู้เชี่ยวชาญ จงสรุปคำตอบจากข้อมูลที่ได้จากฐานข้อมูลกฎหมาย

Question: {question}

Context (ข้อมูลจาก Graph Database):
{context}

Instructions:
- นำเนื้อหาจาก 'd.text' มาตอบคำถามให้ชัดเจน
- หากมีการกล่าวถึงมาตราหรือแนวคิดที่เกี่ยวข้อง (Mentions/Neighbors) ให้สรุปมาด้วย
- ตอบเป็นภาษาไทยที่สุภาพ เป็นทางการ และเข้าใจง่าย
- หากข้อมูลใน Context ไม่เพียงพอ ให้แจ้งว่าไม่พบข้อมูลที่ระบุในระบบ

Answer:"""

MY_QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"], 
    template=QA_TEMPLATE
)

In [8]:
chain = GraphCypherQAChain.from_llm(
    graph=graph,
    # llm= ChatOpenAI(temperature=0, model="anthropic/claude-haiku-4.5", api_key=os.getenv("openrouter_API_key"), base_url="https://openrouter.ai/api/v1"),
    cypher_llm=ChatOpenAI(temperature=0, model="anthropic/claude-haiku-4.5", api_key=os.getenv("openrouter_API_key"), base_url="https://openrouter.ai/api/v1"),
    qa_llm=ChatOpenAI(temperature=0, model="openai/gpt-4o-mini", api_key=os.getenv("openrouter_API_key"), base_url="https://openrouter.ai/api/v1"),
    verbose=True,
    top_k=10,
    return_direct=False,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    qa_prompt=MY_QA_PROMPT,
    # function_response_system="Respond as a bird that is a professional lawyer!",
    allow_dangerous_requests=True,
    # use_function_response=True,
)

In [ ]:
import time 

with get_openai_callback() as cb:
    start_time = time.perf_counter()

    response = chain.invoke({"query": "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"}) # bird lawyer
   
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.4f} seconds")

    print(f"คำถาม: {response['query']}")
    print(f"คำตอบ: {response['result']}")   
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (d:Document)
WHERE d.law_name CONTAINS "สัญญาซื้อขายล่วงหน้า" 
  AND d.text CONTAINS "ประกอบกิจการ" 
  AND d.text CONTAINS "ไม่ได้รับใบอนุญาต"
OPTIONAL MATCH (d)-[:MENTIONS]->(neighbor)
RETURN d.law_name, d.section_num, d.text, 
       collect(DISTINCT {label: labels(neighbor)[0], id: neighbor.id}) AS Mentions

Full Context:
[{'d.law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'd.section_num': '132', 'd.text': 'กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nมาตรา: 132\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน', 'Mentions': [{'id': 'มาตรา 54', 'label': 'Clause'}, {'id': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'label': 'Law'}, {'id

In [10]:
import time

with get_openai_callback() as cb:
    start_time = time.perf_counter()

    response = chain.invoke({"query": "ถ้าผู้ิยู่ในปกครองได้ยินยอมในการกระทำของผู้ปกครองจะทำให้ผู้ปกครองหลุดพ้นจากความรับผิดหรือเปล่า"})
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.4f} seconds")
    print(f"คำถาม: {response['query']}")
    print(f"คำตอบ: {response['result']}")   
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")




> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (d:Document)
WHERE d.text CONTAINS "ผู้อยู่ในปกครอง" 
  AND d.text CONTAINS "ยินยอม" 
  AND d.text CONTAINS "ผู้ปกครอง"
  AND d.text CONTAINS "ความรับผิด"
OPTIONAL MATCH (d)-[:MENTIONS]->(entity:Entity)
OPTIONAL MATCH (d)-[:MENTIONS]->(action:Action)
OPTIONAL MATCH (d)-[:MENTIONS]->(responsibility:Responsibility)
OPTIONAL MATCH (d)-[:MENTIONS]->(consequence:Consequence)
RETURN d.law_name, d.section_num, d.text,
       collect(DISTINCT {label: "Entity", id: entity.id}) AS Entities,
       collect(DISTINCT {label: "Action", id: action.id}) AS Actions,
       collect(DISTINCT {label: "Responsibility", id: responsibility.id}) AS Responsibilities,
       collect(DISTINCT {label: "Consequence", id: consequence.id}) AS Consequences

Full Context:
[{'d.law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 'd.section_num': '1598/5', 'd.text': 'กฎหมาย: ประมวลกฎหมายแพ่งและพาณิชย์\nมาตรา: 1598/5\nเนื้อหา: ประมวลกฎหมายแพ่งและพาณิชย์ มาต

### VectorRetriever

In [56]:
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever, Text2CypherRetriever

In [ ]:
# # Initialize the retriever
# embedder = OpenAIEmbeddings(api_key=openai_api_key, model="")

# retriever = VectorRetriever(
#      driver,
#      index_name= "text_embeddings",
#      embedder=embedder,
#      return_properties=["text"]
# )
# query = "What are the main risks around cryptocurrency?"
# result = retriever.search(query_text=query, top_k=10)

### VectorCypherRetriever

In [ ]:
# vector_cypher_retriever = VectorCypherRetriever(
#     driver=driver,
#     index_name='chunkEmbeddings',
#     embedder=embedder,
#     retrieval_query=chunk_to_asset_manager_query
# )

In [ ]:
# result = GraphRag(llm=llm,retriever=vector_cyper_retriever)
# print(rag.search(query_text=query_text).answer)